MAKE ML MODELS ONLY FOR THE FIRST LEVEL OF COMPLAINT (THE CLEAR ONE) THEN EAVALUATE 
IF RESULT ARE ACCURATE CONTINUE WITH THE SECOND LEVEL AND SO ON

In [74]:
# importer toutes les bib

import pandas 
import re 
import spacy
import numpy
from sklearn.feature_extraction.text import CountVectorizer , TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score , classification_report , confusion_matrix
from sklearn.naive_bayes import MultinomialNB
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.python.keras.layers import Embedding
from tensorflow.python.keras.models import sequential
from keras.preprocessing.sequence import pad_sequences
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [75]:
# importer le dataset et le preparer 
complaint_dataset = pandas.read_csv('DATA\MergedComplaints.csv')
complaint_dataset = complaint_dataset[ ["Commentaire client"  , "Predicted class"] ]
complaint_dataset['Predicted class'] = complaint_dataset['Predicted class'].map( {'Problème / Réclamation':0 , 'Inapproprié':1 , 'Question/Assistance':2 , 'Positif':3 , 'Suggestion':4})

complaint_dataset = complaint_dataset.dropna(subset=["Predicted class"])

print(complaint_dataset["Predicted class"].value_counts())

Predicted class
0.0    18388
1.0     4426
2.0     3316
3.0     3072
4.0      769
Name: count, dtype: int64


<>:2: SyntaxWarning: invalid escape sequence '\M'
<>:2: SyntaxWarning: invalid escape sequence '\M'
C:\Users\Administrator\AppData\Local\Temp\ipykernel_20420\3017492877.py:2: SyntaxWarning: invalid escape sequence '\M'
  complaint_dataset = pandas.read_csv('DATA\MergedComplaints.csv')


In [76]:
complaint_dataset["Commentaire client"] = complaint_dataset["Commentaire client"].fillna("").astype(str)

TF_IDF = TfidfVectorizer( ngram_range=(1,2) , max_features=5000)

X_train , X_test , Y_train , Y_test = train_test_split(
    complaint_dataset["Commentaire client"] , complaint_dataset["Predicted class"] , test_size=0.25
)
X_train_vec = TF_IDF.fit_transform(X_train)
X_test_vec = TF_IDF.transform(X_test)

# pour liberer espace 
import gc
del complaint_dataset
gc.collect()

0

try the logistic regression

In [77]:
logisticModel = LogisticRegression(max_iter=450)
logisticModel.fit(X_train_vec,Y_train)

prediction = logisticModel.predict(X_test_vec)

print( classification_report(Y_test, prediction) )

              precision    recall  f1-score   support

         0.0       0.90      0.98      0.94      4583
         1.0       0.84      0.78      0.81      1113
         2.0       0.92      0.82      0.87       855
         3.0       0.95      0.87      0.91       761
         4.0       0.85      0.26      0.40       181

    accuracy                           0.90      7493
   macro avg       0.89      0.74      0.78      7493
weighted avg       0.90      0.90      0.90      7493



In [78]:
confusion_matrix(Y_test , prediction).tolist()

[[4476, 58, 36, 9, 4],
 [219, 865, 12, 16, 1],
 [112, 36, 703, 2, 2],
 [41, 53, 2, 664, 1],
 [102, 13, 10, 9, 47]]

In [79]:
# 2eme methode : voir si ya amelioration 

from sklearn.linear_model import SGDClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils import shuffle

# 'log_loss' = régression logistique
classes = numpy.array([0,1,2,3,4])
weights = compute_class_weight(
    class_weight = "balanced",
    classes = classes ,
    y =Y_train
)
class_weight = dict(zip(classes,weights))
sgd_model = SGDClassifier(loss='log_loss', class_weight=class_weight)
numIter = 3

for case in range(numIter):
    # rediviser
    X_shuffled, Y_shuffled = shuffle(X_train_vec, Y_train)
    
    # partial_fit permet d'entraîner le modèle sans repartir de zéro
    sgd_model.partial_fit(X_shuffled, Y_shuffled, classes=[0,1,2,3,4])


prediction = sgd_model.predict(X_test_vec)
print(classification_report(Y_test, prediction))

              precision    recall  f1-score   support

         0.0       0.94      0.94      0.94      4583
         1.0       0.79      0.78      0.79      1113
         2.0       0.86      0.84      0.85       855
         3.0       0.90      0.89      0.90       761
         4.0       0.59      0.75      0.66       181

    accuracy                           0.89      7493
   macro avg       0.82      0.84      0.83      7493
weighted avg       0.89      0.89      0.89      7493



In [80]:
confusion_matrix(Y_test , prediction).tolist()

[[4287, 130, 89, 31, 46],
 [163, 869, 24, 32, 25],
 [82, 36, 721, 4, 12],
 [19, 50, 1, 681, 10],
 [26, 9, 1, 9, 136]]

In [81]:
texTry =["غير الاشهار"]
texTry = TF_IDF.transform(texTry)

print(logisticModel.predict(texTry))

[1.]


try the multi naive bais

In [82]:
# on garde le tfifd les memes tests
naiveModel = MultinomialNB()
naiveModel.fit(X_train_vec,Y_train)

prediction2 = naiveModel.predict(X_test_vec)

print( classification_report(Y_test, prediction) )

              precision    recall  f1-score   support

         0.0       0.94      0.94      0.94      4583
         1.0       0.79      0.78      0.79      1113
         2.0       0.86      0.84      0.85       855
         3.0       0.90      0.89      0.90       761
         4.0       0.59      0.75      0.66       181

    accuracy                           0.89      7493
   macro avg       0.82      0.84      0.83      7493
weighted avg       0.89      0.89      0.89      7493



In [83]:
confusion_matrix(Y_test , prediction2).tolist()

[[4486, 46, 25, 24, 2],
 [511, 579, 5, 18, 0],
 [293, 12, 542, 8, 0],
 [101, 15, 2, 643, 0],
 [137, 10, 15, 10, 9]]

In [84]:
# the best result is the logistic regression ameliorated with SGDClassifier who helps the model train by data lots
# because initilay calculating the decent gradient mean using all of the data which is heavy but the SGD calculate the desecent gradient with samples 
# a small increase in efficiency and time